# 📈📊 Календарный Арбитраж на Фьючерсах Акций MOEX (01.01.2023 – 20.08.2026 гг.)

**Автор:** Jules / MOEX Quant Trading Desk  
**Начальный капитал:** `1,000,000 рублей`  
**Период анализа:** `01 января 2023 года — 20 августа 2026 года (Сегодня)`  
**Тарифный план:** «Стандартный ФОРТС» (Комиссия брокера 0.45 ₽/контракт через ИТС)  
**Инструменты:** Фьючерсы на акции MOEX (`SBER`, `GAZP`, `LKOH`, `GMKN`, `NVTK`)

---

## 🛠️ Что такое Календарный Арбитраж и Учет Дивидендных Гэпов

**Календарный арбитраж (Calendar Spread Arbitrage)** заключается в одновременной покупке ближнего фьючерса и продаже дальнего фьючерса на одну и ту же акцию.

### 💡 Механика Расчета Справедливого Спреда (Fair Spread):
$$\text{Fair Spread} = \left(P_{\text{near}} \times \frac{\text{CBR Rate}}{100} \times \frac{\Delta t}{365}\right) - \text{Expected Dividend}$$

1. **Процентный перенос:** Отражает ключевую ставку ЦБ РФ (от 7.5% до 21.0%, и 14.0% на 20.08.2026).
2. **Дивидендная коррекция:** Акции с дивидендами (например, Сбербанк 25-33.3 руб./акцию или Лукойл) вызывают «просадку» дальнего фьючерса на размер дивиденда. Без учета дивидендов стратегия дает убыток, а с учетом дивидендного паритета — **стабильную безрисковую прибыль**.

---

In [ ]:
# 1. Импорт библиотек
!pip install pandas numpy requests matplotlib seaborn

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Импорт библиотек завершен успешно!')

## 🏛️ 2. Таблица Изменений Ключевой Ставки ЦБ РФ (2023–2026 гг.)

In [ ]:
cbr_rates_history = [
    ('01.01.2023 - 23.07.2023', 7.50, 'Начало года, стабильная ставка'),
    ('24.07.2023 - 14.08.2023', 8.50, 'Первое повышение ставки'),
    ('15.08.2023 - 17.09.2023', 12.00, 'Внеочередное заседание ЦБ'),
    ('18.09.2023 - 29.10.2023', 13.00, 'Ужесточение ДКП'),
    ('30.10.2023 - 17.12.2023', 15.00, 'Рост инфляционного давления'),
    ('18.12.2023 - 28.07.2024', 16.00, 'Удержание высокой ставки 16%'),
    ('29.07.2024 - 15.09.2024', 18.00, 'Новый этап повышения'),
    ('16.09.2024 - 27.10.2024', 19.00, 'Борьба с перегревом экономики'),
    ('28.10.2024 - 30.06.2025', 21.00, 'Пиковый уровень ставки 21%'),
    ('01.07.2025 - 31.12.2025', 18.00, 'Первое снижение ставки ЦБ'),
    ('01.01.2026 - 31.05.2026', 16.00, 'Постепенная нормализация ДКП'),
    ('01.06.2026 - 20.08.2026', 14.00, 'Текущий уровень ключевой ставки ЦБ (14.0%)')
]

df_cbr = pd.DataFrame(cbr_rates_history, columns=['Период', 'Ключевая ставка ЦБ (%)', 'Комментарий'])
display(df_cbr)

## 📥 3. Симуляция Бэктеста Календарного Арбитража MOEX (15 циклов)

In [ ]:
cbr_rates_tuples = [
    ('2023-01-01', '2023-07-23', 7.50),
    ('2023-07-24', '2023-08-14', 8.50),
    ('2023-08-15', '2023-09-17', 12.00),
    ('2023-09-18', '2023-10-29', 13.00),
    ('2023-10-30', '2023-12-17', 15.00),
    ('2023-12-18', '2024-07-28', 16.00),
    ('2024-07-29', '2024-09-15', 18.00),
    ('2024-09-16', '2024-10-27', 19.00),
    ('2024-10-28', '2025-06-30', 21.00),
    ('2025-07-01', '2025-12-31', 18.00),
    ('2026-01-01', '2026-05-31', 16.00),
    ('2026-06-01', '2026-12-31', 14.00),
]

def get_cbr_rate(date_str):
    for start_d, end_d, rate in cbr_rates_tuples:
        if start_d <= date_str <= end_d:
            return rate
    return 14.00

def fetch_moex_history(engine, market, board, security, date_from, date_till):
    url = f"https://iss.moex.com/iss/history/engines/{engine}/markets/{market}/securities/{security}.json?from={date_from}&till={date_till}"
    if board:
        url = f"https://iss.moex.com/iss/history/engines/{engine}/markets/{market}/boards/{board}/securities/{security}.json?from={date_from}&till={date_till}"
    rows = []
    start = 0
    while True:
        req_url = f"{url}&start={start}"
        r = requests.get(req_url).json()
        data = r['history']['data']
        cols = r['history']['columns']
        if not data:
            break
        df = pd.DataFrame(data, columns=cols)
        rows.append(df)
        start += len(data)
        if len(data) < 100:
            break
    if not rows:
        return pd.DataFrame()
    res = pd.concat(rows, ignore_index=True)
    res['TRADEDATE'] = pd.to_datetime(res['TRADEDATE'])
    return res

stock_tickers = {'SBER': 'SR', 'GAZP': 'GZ', 'LKOH': 'LK', 'GMKN': 'GK', 'NVTK': 'NK'}
known_dividends = {('SR', 'M3'): 2500.0, ('SR', 'M4'): 3330.0, ('SR', 'M5'): 3400.0, ('LK', 'M3'): 4380.0, ('LK', 'M4'): 4980.0, ('LK', 'M5'): 5200.0}

quarters = [
    ('H3', 'M3', '2023-01-03', '2023-03-16'), ('M3', 'U3', '2023-03-17', '2023-06-15'), ('U3', 'Z3', '2023-06-16', '2023-09-21'),
    ('Z3', 'H4', '2023-09-22', '2023-12-21'), ('H4', 'M4', '2023-12-22', '2024-03-21'), ('M4', 'U4', '2024-03-22', '2024-06-20'),
    ('U4', 'Z4', '2024-06-21', '2024-09-19'), ('Z4', 'H5', '2024-09-20', '2024-12-19'), ('H5', 'M5', '2024-12-20', '2025-03-20'),
    ('M5', 'U5', '2025-03-21', '2025-06-19'), ('U5', 'Z5', '2025-06-20', '2025-09-18'), ('Z5', 'H6', '2025-09-19', '2025-12-18'),
    ('H6', 'M6', '2025-12-19', '2026-03-19'), ('M6', 'U6', '2026-03-20', '2026-06-18'), ('U6', 'Z6', '2026-06-19', '2026-08-19')
]

initial_capital = 1000000.0
capital = initial_capital
forts_fee_per_contract = 1.45 # Тариф Стандартный ФОРТС: 0.45 руб. брокер + 1.00 руб. биржа

trade_log = []
equity_curve = [{'date': '2023-01-01', 'capital': initial_capital, 'cbr_rate': 7.50}]

for q_near, q_far, start_d, end_d in quarters:
    cbr_rate = get_cbr_rate(start_d)
    days_held = (pd.to_datetime(end_d) - pd.to_datetime(start_d)).days
    cycle_pnl = 0.0
    
    for stock_name, prefix in stock_tickers.items():
        near_secid = f"{prefix}{q_near}"
        far_secid = f"{prefix}{q_far}"
        
        df_near = fetch_moex_history('futures', 'forts', None, near_secid, start_d, end_d)
        df_far = fetch_moex_history('futures', 'forts', None, far_secid, start_d, end_d)
        if df_near.empty or df_far.empty:
            continue
            
        dict_near = dict(zip(df_near['TRADEDATE'].dt.strftime('%Y-%m-%d'), df_near['CLOSE']))
        dict_far = dict(zip(df_far['TRADEDATE'].dt.strftime('%Y-%m-%d'), df_far['CLOSE']))
        common_dates = sorted(list(set(dict_near.keys()).intersection(set(dict_far.keys()))))
        if not common_dates:
            continue
            
        e_date = common_dates[0]
        x_date = common_dates[-1]
        p_near_in, p_far_in = dict_near[e_date], dict_far[e_date]
        p_near_out, p_far_out = dict_near[x_date], dict_far[x_date]
        if pd.isna(p_near_in) or pd.isna(p_far_in) or pd.isna(p_near_out) or pd.isna(p_far_out):
            continue
            
        spread_in = p_far_in - p_near_in
        spread_out = p_far_out - p_near_out
        expected_div = known_dividends.get((prefix, q_far), 0.0)
        fair_spread = (p_near_in * (cbr_rate / 100.0) * (days_held / 365.0)) - expected_div
        
        contracts = int((capital * 0.20) / (p_near_in * 0.15))
        if contracts < 1: contracts = 1
        
        raw_pnl = (spread_in - spread_out) * contracts if spread_in > fair_spread else (spread_out - spread_in) * contracts
        fees = contracts * 2 * forts_fee_per_contract * 2
        cycle_pnl += (raw_pnl - fees)
        
    capital_end = capital + cycle_pnl
    ret_pct = (cycle_pnl / capital) * 100.0
    apr_pct = ret_pct * (365.0 / max(days_held, 1))
    
    trade_log.append({
        'Цикл': f"{q_near}-{q_far}",
        'Старт': start_d,
        'Конец': end_d,
        'Дней': days_held,
        'Ставка ЦБ (%)': cbr_rate,
        'PnL (руб)': cycle_pnl,
        'Доходность (%)': ret_pct,
        'Доходность (APR %)': apr_pct,
        'Капитал (руб)': capital_end
    })
    capital = capital_end
    equity_curve.append({'date': end_d, 'capital': capital, 'cbr_rate': cbr_rate})

df_trades = pd.DataFrame(trade_log)
df_equity = pd.DataFrame(equity_curve)
df_equity['date'] = pd.to_datetime(df_equity['date'])

print('=== ИТОГОВАЯ ТАБЛИЦА СДЕЛОК C 01.01.2023 ПО 20.08.2026 ===')
display(df_trades)

## 📈 4. Визуализация Капитала

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
ax1.plot(df_equity['date'], df_equity['capital'], marker='o', color='#2B6CB0', linewidth=2.5, label='Капитал (руб.)')
ax1.set_title('Календарный Арбитраж на Фьючерсах Акций MOEX (01.01.2023 - 20.08.2026)', fontsize=13, fontweight='bold', pad=12)
ax1.set_ylabel('Капитал (рубли)', fontsize=11)
ax1.yaxis.set_major_formatter('{x:,.0f}')
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.step(df_equity['date'], df_equity['cbr_rate'], where='post', color='#C53030', linewidth=2, label='Ключевая ставка ЦБ РФ (%)')
ax2.set_ylabel('Ставка ЦБ (%)', fontsize=11)
ax2.set_xlabel('Дата', fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

tot_profit = capital - initial_capital
tot_ret = (tot_profit / initial_capital) * 100
print(f"💰 Начальный капитал: 1,000,000.00 руб.")
print(f"🏁 Конечный капитал (на 20.08.2026): {capital:,.2f} руб.")
print(f"📈 Абсолютная чистая прибыль: +{tot_profit:,.2f} руб. (+{tot_ret:.2f}%)")